# Advanced Problems with Solutions: Attribute Read Accessors

This notebook is a **tutorial-style problem set** on Python attribute reading, especially:

- `__getattr__`
- `__getattribute__`
- safe delegation with `super()` / `object.__getattribute__`
- recursion hazards
- lazy and dynamic attributes
- descriptors and properties
- proxy objects
- class-level access through metaclasses
- method lookup
- `getattr`, `hasattr`, and `AttributeError` semantics

The goal is not just to show final code. We will build each idea **one small step at a time**, including approaches that look reasonable but are subtly wrong.

The style throughout is deliberately incremental:

1. state the problem,
2. predict what Python will do,
3. try a small experiment,
4. inspect the result,
5. identify the trap,
6. build the solution,
7. verify it with assertions,
8. extract a reusable rule.

> **Important:** Some examples intentionally demonstrate failure modes. Those cells catch the exception so that the notebook can still be executed from top to bottom.

## 0. A compact mental model before the problems

For ordinary instance access such as `obj.name`, Python begins with `obj.__getattribute__('name')`.

If normal lookup ultimately fails with `AttributeError`, Python may then call `obj.__getattr__('name')` **if the class defines it**.

That difference drives almost everything in this notebook:

- `__getattribute__` participates in **every** normal attribute read.
- `__getattr__` is a **fallback** for missing attributes.

So `__getattr__` is usually easier to customize safely, while `__getattribute__` is much more powerful—and much easier to break.

In [1]:
class LookupDemo:
    existing = 'class value'

    def __getattr__(self, name):
        print(f'__getattr__ fallback for {name!r}')
        return '<missing>'

obj = LookupDemo()
print(obj.existing)
print(obj.unknown)

class value
__getattr__ fallback for 'unknown'
<missing>


Notice that `existing` did not trigger `__getattr__`, while `unknown` did.

Keep that observation in mind; several later problems depend on it.

# Problem 1 — A safe private-name fallback

Suppose an object stores data under private-ish names such as `_name` and `_age`, but we want reads of `name` and `age` to fall back to those values when no public attribute exists.

### Step 1 — The tempting implementation

A first attempt might use `getattr(self, '_' + name)` inside `__getattr__`.

That sounds natural: if `name` is missing, try `_name`.

In [2]:
class RecursiveFallback:
    def __getattr__(self, name):
        alt_name = '_' + name
        return getattr(self, alt_name)

r = RecursiveFallback()
try:
    print(r.age)
except RecursionError as ex:
    print(type(ex).__name__, '- recursive lookup detected')

RecursionError - recursive lookup detected


### Step 2 — Why does that recurse?

The missing lookup for `age` enters `__getattr__`.

Inside it, `getattr(self, '_age')` performs another normal attribute lookup. If `_age` is also missing, Python calls `__getattr__` again—this time for `_age`. The process repeats with `__age`, `___age`, and so on.

### Step 3 — Bypass our fallback when performing the internal lookup

Inside the hook, we need a lookup path that does **not** re-enter the same `__getattr__` fallback.

`object.__getattribute__(self, name)` is explicit and makes that intention clear.

In [3]:
class PrivateFallback:
    def __init__(self, **values):
        for key, value in values.items():
            setattr(self, '_' + key, value)

    def __getattr__(self, name):
        alt_name = '_' + name
        try:
            return object.__getattribute__(self, alt_name)
        except AttributeError:
            raise AttributeError(
                f'{type(self).__name__!s} has no attribute {name!r}'
            ) from None

p1 = PrivateFallback(name='Ada', age=36)
print(p1.name)
print(p1.age)

Ada
36


### Step 4 — Verify correct failure semantics

A good fallback should still raise `AttributeError` for truly unknown names. This matters because built-ins such as `hasattr` rely on that exception.

In [4]:
assert p1.name == 'Ada'
assert p1.age == 36
assert hasattr(p1, 'name')
assert not hasattr(p1, 'language')

try:
    p1.language
except AttributeError as ex:
    print(ex)

PrivateFallback has no attribute 'language'


### Solution takeaway

Inside an attribute hook, distinguish **external lookup** from **internal machinery**.

When the hook itself needs to read internal state, prefer a deliberate bypass such as `object.__getattribute__` or an appropriate `super().__getattribute__` call.

# Problem 2 — Lazy attributes that compute once and then behave normally

We want a class where an expensive value is computed only when first requested.

After the first computation, future reads should use an ordinary stored attribute and should **not** call `__getattr__` again.

### Step 1 — Why `__getattr__` is a natural fit

Because `__getattr__` runs only after normal lookup fails, it naturally represents the phrase:

> “If this attribute does not exist yet, create it.”

In [5]:
class LazyReport:
    def __init__(self, numbers):
        self.numbers = list(numbers)
        self.compute_count = 0

    def __getattr__(self, name):
        if name == 'mean':
            self.compute_count += 1
            value = sum(self.numbers) / len(self.numbers)
            setattr(self, name, value)
            return value
        raise AttributeError(name)

report = LazyReport([10, 20, 30, 40])
print('before:', report.__dict__)
print('first mean:', report.mean)
print('after:', report.__dict__)
print('second mean:', report.mean)
print('compute count:', report.compute_count)

before: {'numbers': [10, 20, 30, 40], 'compute_count': 0}
first mean: 25.0
after: {'numbers': [10, 20, 30, 40], 'compute_count': 1, 'mean': 25.0}
second mean: 25.0
compute count: 1


### Step 2 — What changed after the first read?

The first request for `mean` was missing, so `__getattr__` computed it and stored `self.mean`.

The second request finds `mean` through normal lookup. Therefore the fallback is no longer needed.

### Step 3 — Add invalidation

Caching introduces a new design problem: if the source data changes, the cached value may become stale.

We will make that invalidation explicit instead of trying to make the attribute hook guess when mutation matters.

In [6]:
class BetterLazyReport:
    def __init__(self, numbers):
        self.numbers = list(numbers)
        self.compute_count = 0

    def __getattr__(self, name):
        if name == 'mean':
            self.compute_count += 1
            value = sum(self.numbers) / len(self.numbers)
            self.mean = value
            return value
        raise AttributeError(name)

    def add(self, value):
        self.numbers.append(value)
        self.__dict__.pop('mean', None)

report2 = BetterLazyReport([2, 4, 6])
assert report2.mean == 4
assert report2.mean == 4
assert report2.compute_count == 1

report2.add(8)
assert report2.mean == 5
assert report2.compute_count == 2
print(report2.__dict__)

{'numbers': [2, 4, 6, 8], 'compute_count': 2, 'mean': 5.0}


### Solution takeaway

`__getattr__` is excellent for **lazy materialization** because once the attribute is stored, Python's ordinary lookup path takes over.

But cache invalidation remains a separate domain problem and should be designed explicitly.

# Problem 3 — Backward-compatible aliases with deprecation warnings

Imagine an API renamed `surname` to `last_name` and `mail` to `email`.

Old code should keep working temporarily, but users should be warned that the old names are deprecated.

### Step 1 — Put aliases in data, not in a long `if/elif` chain

For a maintainable design, the mapping of old names to new names should be ordinary data.

In [7]:
import warnings

class UserRecord:
    _aliases = {
        'surname': 'last_name',
        'mail': 'email',
    }

    def __init__(self, first_name, last_name, email):
        self.first_name = first_name
        self.last_name = last_name
        self.email = email

### Step 2 — Use `__getattr__` only for names that normal lookup did not find

That is exactly what we want for deprecated aliases: real attributes should win; aliases are only a fallback.

In [8]:
class UserRecord:
    _aliases = {
        'surname': 'last_name',
        'mail': 'email',
    }

    def __init__(self, first_name, last_name, email):
        self.first_name = first_name
        self.last_name = last_name
        self.email = email

    def __getattr__(self, name):
        aliases = object.__getattribute__(self, '_aliases')
        if name in aliases:
            replacement = aliases[name]
            warnings.warn(
                f'{name!r} is deprecated; use {replacement!r}',
                DeprecationWarning,
                stacklevel=2,
            )
            return object.__getattribute__(self, replacement)
        raise AttributeError(name)

### Step 3 — Test both compatibility and normal failure

In [9]:
u = UserRecord('Grace', 'Hopper', 'grace@example.test')

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    old_value = u.surname

print(old_value)
print(str(caught[0].message))
assert old_value == u.last_name
assert not hasattr(u, 'phone')

Hopper
'surname' is deprecated; use 'last_name'


### Step 4 — Why not override `__getattribute__` for this?

We could, but it would run for every read: `first_name`, `email`, `_aliases`, methods, `__dict__`, and everything else.

Since aliases only matter when ordinary lookup fails, `__getattr__` expresses the requirement more directly and with less risk.

### Solution takeaway

Prefer the **least invasive hook** that solves the problem. A missing-name compatibility layer is a textbook `__getattr__` use case.

# Problem 4 — A dynamic configuration object with explicit missing-value behavior

We want to access configuration keys using attribute syntax:

```python
config.host
config.port
```

But the underlying data lives in a dictionary.

### Step 1 — Separate real object attributes from dynamic configuration keys

The object still needs real implementation attributes such as `_data`.

Dynamic lookup should happen only when ordinary attribute lookup has failed.

In [10]:
class ConfigView:
    def __init__(self, mapping):
        self._data = dict(mapping)

    def __getattr__(self, name):
        data = object.__getattribute__(self, '_data')
        try:
            return data[name]
        except KeyError:
            raise AttributeError(
                f'configuration key {name!r} does not exist'
            ) from None

config = ConfigView({'host': 'localhost', 'port': 5432})
print(config.host)
print(config.port)

localhost
5432


### Step 2 — Be careful with dictionary methods

We are **not** forwarding every operation to the dictionary. For example, `config.items` is not automatically `_data.items`.

That is a useful design boundary: configuration keys are exposed; the entire dictionary interface is not.

In [11]:
print('items available?', hasattr(config, 'items'))
print('underlying mapping:', config._data)

items available? False
underlying mapping: {'host': 'localhost', 'port': 5432}


### Step 3 — Add a namespace for nested mappings

Now we will recursively wrap nested dictionaries so that `config.database.host` also works.

In [12]:
class NestedConfig:
    def __init__(self, mapping):
        self._data = dict(mapping)

    def __getattr__(self, name):
        data = object.__getattribute__(self, '_data')
        try:
            value = data[name]
        except KeyError:
            raise AttributeError(name) from None
        if isinstance(value, dict):
            value = type(self)(value)
        return value

cfg = NestedConfig({
    'database': {'host': 'db.local', 'port': 6432},
    'debug': True,
})

print(cfg.database.host)
print(cfg.database.port)
print(cfg.debug)

db.local
6432
True


### Step 4 — Think about identity and caching

Every access to `cfg.database` currently creates a new wrapper object. That may be fine for a read-only view, but it may be surprising if identity matters.

A production design could cache nested wrappers, but that would introduce invalidation and mutation questions similar to the previous problem.

### Solution takeaway

Attribute syntax over mappings is convenient, but it changes error behavior and discoverability. Preserve `AttributeError` for missing attributes and define clearly which dictionary behavior is intentionally exposed.

# Problem 5 — Auditing every attribute read without causing recursion

Now we need something `__getattr__` cannot do: log **every** attribute read, including attributes that already exist.

That means we need `__getattribute__`.

### Step 1 — The classic mistake

Inside `__getattribute__`, even reading `self.log` is itself an attribute read.

So this implementation is dangerous:

```python
def __getattribute__(self, name):
    self.log.append(name)
    ...
```

The attempt to read `self.log` re-enters `__getattribute__` before the original lookup has finished.

### Step 2 — Read internal machinery through the base implementation

In [13]:
class AuditedObject:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self._read_log = []

    def __getattribute__(self, name):
        if name != '_read_log':
            log = object.__getattribute__(self, '_read_log')
            log.append(name)
        return object.__getattribute__(self, name)

    def read_log(self):
        return list(object.__getattribute__(self, '_read_log'))

a = AuditedObject(10, 20)
print(a.x)
print(a.y)
print(a.read_log())

10
20
['x', 'y', 'read_log']


### Step 3 — Notice that method access is logged too

The expression `a.read_log()` first has to retrieve the method attribute `read_log` from `a`. That lookup also passes through `__getattribute__`.

In [14]:
assert 'x' in a.read_log()
assert 'y' in a.read_log()
assert 'read_log' in a.read_log()
print(a.read_log())

['x', 'y', 'read_log', 'read_log', 'read_log', 'read_log', 'read_log']


### Step 4 — Refine the policy

Real auditing often needs exclusions for implementation details. Here we will skip all names beginning with `_` to avoid noisy internal events.

In [15]:
class PublicReadAudit:
    def __init__(self, value):
        self.value = value
        self._events = []

    def __getattribute__(self, name):
        if not name.startswith('_'):
            events = object.__getattribute__(self, '_events')
            events.append(name)
        return object.__getattribute__(self, name)

    def events(self):
        return list(object.__getattribute__(self, '_events'))

pa = PublicReadAudit(99)
_ = pa.value
_ = pa.events()
print(pa.events())

['value', 'events', 'events']


### Solution takeaway

With `__getattribute__`, assume that **every read can re-enter your hook** unless you explicitly bypass it.

Keep the hook small, deterministic, and very deliberate about internal state access.

# Problem 6 — A read-only proxy with controlled delegation

We want a proxy object that exposes reads from a target object but blocks selected internal names.

This is more subtle than a dictionary wrapper because methods, properties, and descriptors live on the target's type.

### Step 1 — Start with fallback delegation

A simple proxy can let its own attributes resolve normally and delegate missing names to the target with `__getattr__`.

In [16]:
class Service:
    def __init__(self, name):
        self.name = name
        self._token = 'secret-token'

    def greet(self):
        return f'hello from {self.name}'

class SimpleProxy:
    def __init__(self, target):
        self._target = target

    def __getattr__(self, name):
        target = object.__getattribute__(self, '_target')
        return getattr(target, name)

sp = SimpleProxy(Service('api'))
print(sp.name)
print(sp.greet())

api
hello from api


### Step 2 — Add a read policy

We want target names beginning with `_` to remain inaccessible through the proxy.

The proxy's own `_target` still has to remain internally accessible.

In [17]:
class ReadOnlyProxy:
    def __init__(self, target):
        object.__setattr__(self, '_target', target)

    def __getattr__(self, name):
        if name.startswith('_'):
            raise AttributeError(f'private target attribute {name!r} is hidden')
        target = object.__getattribute__(self, '_target')
        return getattr(target, name)

    def __setattr__(self, name, value):
        raise AttributeError('proxy is read-only')

proxy = ReadOnlyProxy(Service('payments'))
print(proxy.name)
print(proxy.greet())

try:
    print(proxy._token)
except AttributeError as ex:
    print(ex)

try:
    proxy.name = 'changed'
except AttributeError as ex:
    print(ex)

payments
hello from payments
private target attribute '_token' is hidden
proxy is read-only


### Step 3 — A crucial special-method caveat

Python performs many special operations—such as `len(obj)`, iteration, arithmetic, and context-manager lookup—through **special method lookup on the type**, not through ordinary instance fallback in the same way as `obj.some_name`.

So a generic `__getattr__` proxy does not automatically proxy every Python protocol.

In [18]:
class Bucket:
    def __init__(self, items):
        self.items = list(items)

    def __len__(self):
        return len(self.items)

bucket_proxy = ReadOnlyProxy(Bucket([1, 2, 3]))
print('target length:', len(object.__getattribute__(bucket_proxy, '_target')))
try:
    print(len(bucket_proxy))
except TypeError as ex:
    print('proxy length fails:', ex)

target length: 3
proxy length fails: object of type 'ReadOnlyProxy' has no len()


### Solution takeaway

`__getattr__` is good for ordinary delegated names, but a fully transparent proxy requires explicit support for the special protocols you intend to proxy.

# Problem 7 — Descriptor precedence: what actually wins?

Advanced attribute lookup is impossible to understand fully without descriptors.

We will compare:

- a **data descriptor** (`__get__` + `__set__`),
- an instance dictionary entry,
- a **non-data descriptor** (`__get__` only).

### Step 1 — Build tracing descriptors

In [19]:
class DataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        print('DataDescriptor.__get__')
        return '<from data descriptor>'

    def __set__(self, instance, value):
        print('DataDescriptor.__set__', value)
        instance.__dict__['_stored_data'] = value

class NonDataDescriptor:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        print('NonDataDescriptor.__get__')
        return '<from non-data descriptor>'

class DescriptorDemo:
    data = DataDescriptor()
    nondata = NonDataDescriptor()

### Step 2 — Put competing values in the instance dictionary

In [20]:
d = DescriptorDemo()
d.__dict__['data'] = '<instance data>'
d.__dict__['nondata'] = '<instance nondata>'

print('d.data    ->', d.data)
print('d.nondata ->', d.nondata)

DataDescriptor.__get__
d.data    -> <from data descriptor>
d.nondata -> <instance nondata>


### Step 3 — Interpret the result

The data descriptor wins over the instance dictionary for `data`.

The instance dictionary wins over the non-data descriptor for `nondata`.

A useful simplified order for ordinary instance lookup is:

1. data descriptor on the class,
2. instance dictionary,
3. non-data descriptor or other class attribute,
4. fallback to `__getattr__` if normal lookup raises `AttributeError`.

### Step 4 — Observe how custom `__getattribute__` surrounds that machinery

In [21]:
class TracedDescriptorDemo(DescriptorDemo):
    def __getattribute__(self, name):
        print(f'__getattribute__ sees {name!r}')
        return super().__getattribute__(name)

td = TracedDescriptorDemo()
td.__dict__['nondata'] = 'instance wins'
print(td.data)
print(td.nondata)

__getattribute__ sees '__dict__'
__getattribute__ sees 'data'
DataDescriptor.__get__
<from data descriptor>
__getattribute__ sees 'nondata'
instance wins


### Solution takeaway

A safe custom `__getattribute__` often delegates to `super().__getattribute__` precisely because the base implementation already knows how to apply descriptor precedence correctly.

# Problem 8 — Properties, hidden storage, and an access policy

We want to deny direct reads of single-underscore attributes while still exposing selected values through properties.

This resembles a common pattern: implementation data is private-ish, public properties form the supported interface.

### Step 1 — Write the access rule

In [22]:
class ProtectedProfile:
    def __init__(self, name, age):
        self._name = name
        self._age = age

    def __getattribute__(self, name):
        if name.startswith('_') and not name.startswith('__'):
            raise AttributeError(f'direct access to {name!r} is forbidden')
        return super().__getattribute__(name)

### Step 2 — The obvious property is still wrong

This property would fail:

```python
@property
def name(self):
    return self._name
```

Why? Because `self._name` itself goes through the custom `__getattribute__`, which rejects `_name`.

### Step 3 — Let the property deliberately bypass the public policy

In [23]:
class ProtectedProfile:
    def __init__(self, name, age):
        self._name = name
        self._age = age

    def __getattribute__(self, name):
        if name.startswith('_') and not name.startswith('__'):
            raise AttributeError(f'direct access to {name!r} is forbidden')
        return super().__getattribute__(name)

    @property
    def name(self):
        return object.__getattribute__(self, '_name')

    @property
    def age(self):
        return object.__getattribute__(self, '_age')

profile = ProtectedProfile('Lin', 28)
print(profile.name, profile.age)
try:
    print(profile._name)
except AttributeError as ex:
    print(ex)

Lin 28
direct access to '_name' is forbidden


### Step 4 — Clarify what this does *not* mean

This is an API-level access policy, not a security boundary. Python code can intentionally bypass it using low-level access such as `object.__getattribute__(profile, '_name')`.

The value of the pattern is interface discipline, not hostile-code isolation.

### Solution takeaway

If a custom lookup policy blocks internal attributes, properties and other trusted implementation code may need an explicit bypass path.

# Problem 9 — `hasattr` can hide more than you think

`hasattr(obj, 'name')` conceptually tries `getattr(obj, 'name')` and returns `False` if that lookup raises `AttributeError`.

That means an accidental `AttributeError` raised *inside* a property or hook can make an existing attribute appear absent.

### Step 1 — Create a property with an internal bug

In [24]:
class MisleadingProperty:
    @property
    def total(self):
        # Pretend this was a typo in production code.
        return self.missing_internal_value + 1

mp = MisleadingProperty()
print('has total?', hasattr(mp, 'total'))
try:
    print(mp.total)
except AttributeError as ex:
    print('direct read:', ex)

has total? False
direct read: 'MisleadingProperty' object has no attribute 'missing_internal_value'


### Step 2 — Why is this dangerous?

The class *does* define `total`, but evaluating the property raises `AttributeError`. `hasattr` interprets that as “attribute missing”.

So `AttributeError` has protocol-level meaning. Raise it intentionally for missing attributes; avoid using it as a generic internal error signal.

### Step 3 — Use domain-appropriate exceptions internally

In [25]:
class ClearerProperty:
    def __init__(self, values=None):
        self.values = values

    @property
    def total(self):
        if self.values is None:
            raise RuntimeError('values have not been loaded')
        return sum(self.values)

cp = ClearerProperty()
try:
    print(hasattr(cp, 'total'))
except RuntimeError as ex:
    print('the real problem is visible:', ex)

the real problem is visible: values have not been loaded


### Solution takeaway

`AttributeError` is not just another exception when you customize attribute lookup. It participates directly in Python's lookup protocol and in helpers such as `hasattr`.

# Problem 10 — Method lookup and callable wrapping

A method call such as `obj.process(10)` begins by retrieving the attribute `process` from `obj`.

So `__getattribute__` can observe method retrieval too.

### Step 1 — Trace method access

In [26]:
class MethodTrace:
    def __getattribute__(self, name):
        print(f'lookup -> {name}')
        return super().__getattribute__(name)

    def greet(self, who):
        return f'hello {who}'

mt = MethodTrace()
print(mt.greet('world'))

lookup -> greet
hello world


### Step 2 — Advanced requirement: wrap public methods with call logging

We can inspect the value returned by normal lookup. If it is callable and public, we can return a wrapper.

But we must be careful not to wrap our own internal helpers repeatedly or destroy useful metadata.

In [27]:
from functools import wraps

class MethodCallAudit:
    def __init__(self):
        self._calls = []

    def __getattribute__(self, name):
        value = super().__getattribute__(name)
        if name.startswith('_') or not callable(value):
            return value

        @wraps(value)
        def wrapper(*args, **kwargs):
            calls = object.__getattribute__(self, '_calls')
            calls.append((name, args, kwargs))
            return value(*args, **kwargs)

        return wrapper

    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b

    def call_log(self):
        return list(object.__getattribute__(self, '_calls'))

mca = MethodCallAudit()
print(mca.add(2, 3))
print(mca.multiply(4, 5))
print(mca.call_log())

5
20
[('add', (2, 3), {}), ('multiply', (4, 5), {}), ('call_log', (), {})]


### Step 3 — Identify a subtle cost

Each attribute read of `mca.add` creates a **new wrapper function**. If wrapper identity or performance matters, a different design—such as decorators at class definition time—may be better.

Attribute hooks are powerful, but power is not the same as appropriateness.

### Solution takeaway

Method calls participate in attribute lookup, so accessors can intercept method retrieval. Use that capability sparingly; ordinary decorators are often simpler when the set of methods is known ahead of time.

# Problem 11 — Class-level fallbacks with a metaclass

So far our hooks have customized instance access.

What if we want this behavior?

```python
FeatureFlags.beta
FeatureFlags.unknown_flag
```

Those are reads on the **class object itself**.

### Step 1 — Remember that classes are objects too

A class is an instance of its metaclass. Therefore class-level `__getattr__` behavior belongs on the metaclass.

In [28]:
class FlagMeta(type):
    def __getattr__(cls, name):
        if name.startswith('is_'):
            return False
        raise AttributeError(name)

class FeatureFlags(metaclass=FlagMeta):
    is_stable = True

print(FeatureFlags.is_stable)
print(FeatureFlags.is_experimental)

True
False


### Step 2 — Add class-level lazy materialization

We can even cache a computed class attribute by setting it on the class object from the metaclass hook.

In [29]:
class RegistryMeta(type):
    def __getattr__(cls, name):
        if name.startswith('plugin_'):
            value = f'<loaded {name}>'
            setattr(cls, name, value)
            return value
        raise AttributeError(name)

class PluginRegistry(metaclass=RegistryMeta):
    pass

print(PluginRegistry.plugin_csv)
print(PluginRegistry.plugin_csv)
print('cached in class dict?', 'plugin_csv' in PluginRegistry.__dict__)

<loaded plugin_csv>
<loaded plugin_csv>
cached in class dict? True


### Step 3 — Verify instance lookup is a different path

In [30]:
registry_instance = PluginRegistry()
print('class fallback exists:', PluginRegistry.plugin_json)
print('instance sees cached class attr:', registry_instance.plugin_json)
try:
    print(registry_instance.plugin_xml)
except AttributeError as ex:
    print('instance lookup did not invoke metaclass __getattr__:', ex)

class fallback exists: <loaded plugin_json>
instance sees cached class attr: <loaded plugin_json>
instance lookup did not invoke metaclass __getattr__: 'PluginRegistry' object has no attribute 'plugin_xml'


### Solution takeaway

To customize access **on the class object**, implement the hook on the metaclass. Instance hooks and class-object hooks are related ideas but operate at different levels.

# Problem 12 — Dynamic attributes that remain discoverable with `dir()`

A common usability problem with dynamic attributes is discoverability.

If `obj.foo` works only because of `__getattr__`, an IDE or a human using `dir(obj)` may not know that `foo` is meaningful.

### Step 1 — Observe the mismatch

In [31]:
class DynamicMetrics:
    def __init__(self, metrics):
        self._metrics = dict(metrics)

    def __getattr__(self, name):
        metrics = object.__getattribute__(self, '_metrics')
        if name in metrics:
            return metrics[name]
        raise AttributeError(name)

metrics = DynamicMetrics({'latency_ms': 12.5, 'requests': 1000})
print(metrics.latency_ms)
print('latency_ms in dir?', 'latency_ms' in dir(metrics))

12.5
latency_ms in dir? False


### Step 2 — Teach `dir()` about the dynamic namespace

In [32]:
class DiscoverableMetrics:
    def __init__(self, metrics):
        self._metrics = dict(metrics)

    def __getattr__(self, name):
        metrics = object.__getattribute__(self, '_metrics')
        if name in metrics:
            return metrics[name]
        raise AttributeError(name)

    def __dir__(self):
        ordinary = set(super().__dir__())
        metrics = object.__getattribute__(self, '_metrics')
        return sorted(ordinary | set(metrics))

dm = DiscoverableMetrics({'latency_ms': 12.5, 'requests': 1000})
print('latency_ms in dir?', 'latency_ms' in dir(dm))
print('requests in dir?', 'requests' in dir(dm))

latency_ms in dir? True
requests in dir? True


### Step 3 — Why this matters

Dynamic APIs are easier to use when runtime behavior, introspection, documentation, and editor tooling tell a consistent story.

`__dir__` is not required for `__getattr__`, but it can make a dynamic interface much more humane.

### Solution takeaway

If you expose a meaningful dynamic attribute namespace, consider whether introspection should expose the same namespace.

# Problem 13 — `__slots__` changes where state lives

Earlier examples sometimes inspected `__dict__`. But not every object has one.

Classes using `__slots__` can store attributes without a normal per-instance dictionary.

### Step 1 — A slotted class

In [33]:
class SlottedPoint:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

point = SlottedPoint(3, 4)
print(point.x, point.y)
print('has __dict__?', hasattr(point, '__dict__'))

3 4
has __dict__? False


### Step 2 — Why direct `self.__dict__` lookup is not a general solution

If an attribute hook tries to bypass recursion by reaching into `self.__dict__`, it assumes storage that may not exist and it also ignores descriptors and class attributes.

Delegating to the base lookup machinery is more general.

### Step 3 — Use a slotted fallback safely

In [34]:
class SlottedAlias:
    __slots__ = ('_value',)

    def __init__(self, value):
        self._value = value

    def __getattr__(self, name):
        if name == 'value':
            return object.__getattribute__(self, '_value')
        raise AttributeError(name)

sa = SlottedAlias(42)
print(sa.value)
assert sa.value == 42
assert not hasattr(sa, '__dict__')

42


### Solution takeaway

`__dict__` is an implementation detail available on many objects, not the definition of Python attribute storage. Prefer the language's lookup machinery unless you specifically need dictionary-level behavior.

# Problem 14 — Final integration: a layered settings object

We will combine several ideas into one object while keeping the responsibilities separated.

Requirements:

- real attributes should work normally,
- selected old names should map to new names,
- unknown names should optionally come from a backing dictionary,
- one expensive setting should be computed lazily and cached,
- missing names must still raise `AttributeError`,
- dynamic names should appear in `dir()`,
- internal reads must not recurse.

### Step 1 — Decide the lookup layers before writing code

We want this order:

1. ordinary Python lookup,
2. if missing, check deprecated aliases,
3. if still missing, check dynamic backing data,
4. if the special lazy name was requested, compute and store it,
5. otherwise raise `AttributeError`.

Notice that step 1 is automatically handled before `__getattr__` is called.

### Step 2 — Build the class

In [35]:
class LayeredSettings:
    _aliases = {
        'db_host': 'database_host',
        'db_port': 'database_port',
    }

    def __init__(self, **data):
        self._data = dict(data)
        self._computed = 0

    def __getattr__(self, name):
        aliases = object.__getattribute__(self, '_aliases')
        data = object.__getattribute__(self, '_data')

        if name in aliases:
            name = aliases[name]

        if name in data:
            return data[name]

        if name == 'connection_string':
            host = data.get('database_host', 'localhost')
            port = data.get('database_port', 5432)
            value = f'postgresql://{host}:{port}'
            object.__setattr__(self, 'connection_string', value)
            count = object.__getattribute__(self, '_computed')
            object.__setattr__(self, '_computed', count + 1)
            return value

        raise AttributeError(
            f'{type(self).__name__} has no setting {name!r}'
        )

    def __dir__(self):
        ordinary = set(super().__dir__())
        data = object.__getattribute__(self, '_data')
        aliases = object.__getattribute__(self, '_aliases')
        dynamic = set(data) | set(aliases) | {'connection_string'}
        return sorted(ordinary | dynamic)

### Step 3 — Exercise every lookup path

In [36]:
settings = LayeredSettings(
    database_host='db.internal',
    database_port=6543,
    debug=True,
)

print('normal dynamic key:', settings.debug)
print('new name:', settings.database_host)
print('old alias:', settings.db_host)
print('lazy:', settings.connection_string)
print('lazy again:', settings.connection_string)
print('computed:', settings._computed)
print('discoverable?', 'connection_string' in dir(settings))

normal dynamic key: True
new name: db.internal
old alias: db.internal
lazy: postgresql://db.internal:6543
lazy again: postgresql://db.internal:6543
computed: 1
discoverable? True


### Step 4 — Verify invariants with assertions

In [37]:
assert settings.debug is True
assert settings.database_host == 'db.internal'
assert settings.db_host == 'db.internal'
assert settings.db_port == 6543
assert settings.connection_string == 'postgresql://db.internal:6543'
assert settings._computed == 1
assert 'debug' in dir(settings)
assert 'db_host' in dir(settings)
assert not hasattr(settings, 'does_not_exist')
print('all integration assertions passed')

all integration assertions passed


### Step 5 — Why this solution uses `__getattr__`, not `__getattribute__`

Every feature in the requirements is about what to do **after an ordinary name is missing**.

Using `__getattribute__` would force us to intercept every read even though most reads need no special behavior. The less invasive hook keeps the implementation easier to reason about.

### Final integration takeaway

Advanced attribute access becomes manageable when you define a clear lookup policy first and then choose the smallest hook that implements that policy.

# Debugging Lab — Diagnose the hook before fixing it

The next problems are intentionally shorter. For each one, read the code first and predict the failure mode before looking at the solution.

## Debugging Problem A — Recursive audit

What is wrong with this design?

```python
class BrokenAudit:
    def __init__(self):
        self.log = []

    def __getattribute__(self, name):
        self.log.append(name)
        return super().__getattribute__(name)
```

### Solution

Reading `self.log` invokes `__getattribute__` again, which tries to read `self.log` again, and so on.

The fix is to obtain the log using the base implementation before appending.

In [38]:
class FixedAudit:
    def __init__(self):
        self.log = []
        self.value = 10

    def __getattribute__(self, name):
        if name != 'log':
            log = object.__getattribute__(self, 'log')
            log.append(name)
        return object.__getattribute__(self, name)

fa = FixedAudit()
print(fa.value)
print(fa.log)

10
['value']


## Debugging Problem B — Swallowing too much

Consider a `__getattr__` that catches `Exception` around complicated code and then raises `AttributeError`.

Why is that dangerous?

### Solution

It can transform unrelated bugs—`TypeError`, `ValueError`, database failures, parsing errors—into “attribute missing”. That makes debugging harder and can mislead `hasattr`.

Catch only the specific exception that actually means the dynamic name is absent.

In [39]:
class PreciseLookup:
    def __init__(self, data):
        self._data = dict(data)

    def __getattr__(self, name):
        data = object.__getattribute__(self, '_data')
        try:
            return data[name]
        except KeyError:
            raise AttributeError(name) from None

pl = PreciseLookup({'answer': 42})
assert pl.answer == 42
assert not hasattr(pl, 'missing')
print('precise exception translation works')

precise exception translation works


## Debugging Problem C — Returning `None` for every missing name

This is tempting:

```python
def __getattr__(self, name):
    return None
```

What protocol does it break?

### Solution

It makes **every possible attribute appear to exist** from the perspective of `hasattr`, because no `AttributeError` is raised.

In [40]:
class NoneForMissing:
    def __getattr__(self, name):
        return None

nfm = NoneForMissing()
print('has completely_fake_name?', hasattr(nfm, 'completely_fake_name'))
print('value:', nfm.completely_fake_name)

has completely_fake_name? True
value: None


Sometimes that behavior is intentionally desired, but it should be a conscious API choice because it changes the meaning of attribute existence.

# Bonus Challenge 1 — A namespace with computed expressions

Build a namespace where ordinary values live in a mapping, while selected names are computed from functions.

For example, `stats.average` should call a registered computation only when requested.

### Step 1 — Represent values and computations separately

In [41]:
class ComputedNamespace:
    def __init__(self, values, computations):
        self._values = dict(values)
        self._computations = dict(computations)

    def __getattr__(self, name):
        values = object.__getattribute__(self, '_values')
        computations = object.__getattribute__(self, '_computations')

        if name in values:
            return values[name]
        if name in computations:
            return computations[name](values)
        raise AttributeError(name)

stats = ComputedNamespace(
    values={'a': 10, 'b': 20, 'c': 30},
    computations={
        'average': lambda d: sum(d.values()) / len(d),
        'maximum': lambda d: max(d.values()),
    },
)

print(stats.a)
print(stats.average)
print(stats.maximum)

10
20.0
30


### Step 2 — Decide whether computed attributes should be cached

This version intentionally does **not** cache. If the backing mapping changes, the computed result changes automatically.

That is a different tradeoff from the lazy-cache problem earlier.

In [42]:
object.__getattribute__(stats, '_values')['a'] = 100
print('new average:', stats.average)

new average: 50.0


### Bonus takeaway

“Dynamic attribute” does not imply “cache it”. Decide whether the dynamic name represents a stable materialized value or a live view over changing state.

# Bonus Challenge 2 — A strict proxy that distinguishes proxy attributes from target attributes

We want the proxy to expose its own diagnostic method while forwarding ordinary public names to the target.

### Step 1 — Let normal proxy attributes win

Because `__getattr__` is only called after normal lookup fails, methods defined on the proxy naturally take precedence over delegated target attributes.

In [43]:
class DiagnosticProxy:
    def __init__(self, target):
        self._target = target
        self._forwarded = []

    def proxy_status(self):
        forwarded = object.__getattribute__(self, '_forwarded')
        return {'forwarded_reads': list(forwarded)}

    def __getattr__(self, name):
        if name.startswith('_'):
            raise AttributeError(name)
        target = object.__getattribute__(self, '_target')
        forwarded = object.__getattribute__(self, '_forwarded')
        forwarded.append(name)
        return getattr(target, name)

class Target:
    value = 123
    def ping(self):
        return 'pong'

proxy2 = DiagnosticProxy(Target())
print(proxy2.value)
print(proxy2.ping())
print(proxy2.proxy_status())

123
pong
{'forwarded_reads': ['value', 'ping']}


### Step 2 — Observe the clean layering

`proxy_status` is a real proxy method, so normal lookup finds it before `__getattr__` runs. Only missing public names are delegated.

# Bonus Challenge 3 — Predict the lookup order

Before running the next cell, predict which messages appear and in what order.

In [44]:
class LoudDescriptor:
    def __get__(self, instance, owner):
        print('2. descriptor __get__')
        return 'descriptor value'

class OrderDemo:
    item = LoudDescriptor()

    def __getattribute__(self, name):
        print(f'1. __getattribute__({name!r})')
        return super().__getattribute__(name)

    def __getattr__(self, name):
        print(f'3. __getattr__({name!r})')
        return 'fallback value'

od = OrderDemo()
print('existing descriptor ->', od.item)
print('missing attribute   ->', od.other)

1. __getattribute__('item')
2. descriptor __get__
existing descriptor -> descriptor value
1. __getattribute__('other')
3. __getattr__('other')
missing attribute   -> fallback value


### Solution

For `od.item`, custom `__getattribute__` runs first and then delegates to normal lookup, which invokes the descriptor.

For `od.other`, custom `__getattribute__` runs first, delegated normal lookup fails with `AttributeError`, and only then does Python invoke `__getattr__`.

# Independent Practice — Try before revealing the solution

These are shorter exercises. Each one includes a starter cell followed by a worked solution.

## Practice 1 — Case-insensitive dynamic fields

Implement a class whose backing dictionary uses lowercase keys, while attribute reads may use any case.

Requirements:

- `obj.Name`, `obj.NAME`, and `obj.name` should all find key `name`.
- truly missing names must raise `AttributeError`.
- internal state access must not recurse.

In [45]:
# Starter sketch — complete this before reading the solution.
class CaseInsensitiveFieldsStarter:
    def __init__(self, mapping):
        self._data = {str(k).lower(): v for k, v in mapping.items()}

    # TODO: implement __getattr__
    pass

### Worked solution

In [46]:
class CaseInsensitiveFields:
    def __init__(self, mapping):
        self._data = {str(k).lower(): v for k, v in mapping.items()}

    def __getattr__(self, name):
        data = object.__getattribute__(self, '_data')
        key = name.lower()
        try:
            return data[key]
        except KeyError:
            raise AttributeError(name) from None

ci = CaseInsensitiveFields({'Name': 'Ada', 'AGE': 36})
assert ci.name == 'Ada'
assert ci.NAME == 'Ada'
assert ci.Age == 36
assert not hasattr(ci, 'country')
print('case-insensitive lookup passed')

case-insensitive lookup passed


## Practice 2 — Count only successful public reads

Implement `__getattribute__` so that successful reads of public attributes increment a counter, but failed reads do not.

Hint: you need to delegate first, then increment only after lookup succeeds.

In [47]:
# Starter sketch
class SuccessfulReadCounterStarter:
    def __init__(self):
        self.value = 10
        self._read_count = 0

    # TODO: implement __getattribute__
    pass

### Worked solution

In [48]:
class SuccessfulReadCounter:
    def __init__(self):
        self.value = 10
        self._read_count = 0

    def __getattribute__(self, name):
        value = object.__getattribute__(self, name)  # may raise
        if not name.startswith('_'):
            count = object.__getattribute__(self, '_read_count')
            object.__setattr__(self, '_read_count', count + 1)
        return value

src = SuccessfulReadCounter()
print(src.value)
try:
    src.missing
except AttributeError:
    pass
print('successful public reads:', src._read_count)
assert src._read_count == 1

10
successful public reads: 1


## Practice 3 — A default factory instead of one fixed default

Create a class where missing attributes are created using a factory function that receives the missing name.

Example: requesting `obj.alpha` could create `'generated:alpha'`.

In [49]:
# Starter sketch
class FactoryDefaultsStarter:
    def __init__(self, factory):
        self._factory = factory

    # TODO: implement __getattr__
    pass

### Worked solution

In [50]:
class FactoryDefaults:
    def __init__(self, factory):
        self._factory = factory

    def __getattr__(self, name):
        factory = object.__getattribute__(self, '_factory')
        value = factory(name)
        setattr(self, name, value)
        return value

fd = FactoryDefaults(lambda name: f'generated:{name}')
print(fd.alpha)
print(fd.beta)
assert fd.alpha == 'generated:alpha'
assert 'alpha' in fd.__dict__

generated:alpha
generated:beta


# Summary — Rules worth remembering

After all of these examples, the most reusable rules are:

1. Use `__getattr__` when the special behavior is only for **missing** names.
2. Use `__getattribute__` only when you truly need to participate in **every** read.
3. Inside hooks, use `object.__getattribute__` or deliberate `super()` delegation for internal state when normal lookup would recurse.
4. Preserve `AttributeError` semantics for genuinely missing attributes.
5. Remember that properties, descriptors, and methods are all part of attribute lookup.
6. A metaclass is the corresponding place to customize reads on the **class object**.
7. Dynamic attributes may need `__dir__` if discoverability matters.
8. Do not assume every object stores state in `__dict__`; `__slots__` and descriptors are counterexamples.
9. Generic proxies do not automatically proxy all special methods.
10. Choose the smallest hook that matches the requirement; simpler mechanisms are easier to reason about.

A good way to practice further is to take any class that exposes data dynamically and ask four questions:

- Which names are real and which are synthesized?
- When should normal lookup win?
- What exception should mean “missing”?
- Which internal reads must bypass the custom hook?

If those answers are clear, the implementation usually becomes much simpler.